# 01 · 工作流模式（6 种 LangGraph 实现）

**对应章节**：LangGraph 教程第 01 章 —— 从「为什么需要工作流」到六种基础模式。

**本 notebook 覆盖 6 种模式**，节点按需共用：
1. Prompt Chain（顺序图）
2. Generator-Evaluator（生成-评估循环）
3. Orchestrator-Worker（并行派发）
4. Router（分类分发）
5. 嵌套子图（Nested Subgraph）
6. Agent（自主循环）

**运行前置**：
- 需要 `.env`（OPENAI_API_KEY / OPENAI_API_BASE / OPENAI_MODEL），放在仓库根目录；
- 需要已 `uv sync`，notebook 内核选 `agent-cookbook`。
- ⚠️ 除第 3 节 Orchestrator-Worker（worker 只打印、不调模型）外，其余模式都会真实调用模型，需可用 API Key。

## 公共头部：引入 LLM 客户端

后续所有依赖 `structured` 的单元都从这里引入。先把仓库根目录加入 `sys.path`，再从 `src.agent_cookbook` 引入 `structured`。`

In [11]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))   # 仓库根目录，使 src 包可被导入
from src.agent_cookbook import structured

## 共享节点：被 6 种模式复用

下面这四个节点 + 两个模型结构在多种模式里反复出现，集中定义一次，后面直接引用：

- `genJoke`：根据 `topic` 生成笑话（带 `reviewResult` 重试提示）；
- `reviewJoke`：当评委，判断笑话是否够好；
- `checkReviewResult`：条件边函数，决定「重试生成」还是「进入翻译」；
- `translate`：把笑话翻成中文；
- 配套 `Joke` / `CriticResult` 两个 pydantic 结构，以及共享的 `AgentState`。

In [ ]:
from typing import TypedDict, Optional, Literal
from pydantic import BaseModel, Field
from langgraph.graph import START, END, StateGraph
from langgraph.config import get_config
from dotenv import load_dotenv
load_dotenv()


class Joke(BaseModel):
    joke: str


class CriticResult(BaseModel):
    isFunny: bool = Field(description="is joke funny or not")
    opinion: str = Field(description="how to improve")


class AgentState(TypedDict):
    topic: str
    content: str
    reviewResult: Optional[CriticResult]
    retryCount: int


def genJoke(state: AgentState):
    message: str = f'gen a Joke about {state["topic"]}'
    if state["reviewResult"]:
        message = message + f', consider the opinion: {state["reviewResult"].opinion}'
    print(f'# gen \n gen message: {message}')
    response = structured(Joke, [{"role":"user","content":message}])
    return {"content": response.joke}


def reviewJoke(state: AgentState):
    response = structured(CriticResult, [
            {"role":"system","content":"You are a strict joke reviewer."},
            {"role":"user","content":f'is this Joke funny or not ? Joke: {state["content"]}, if not let me know how to improve it'}
        ])
    print(f'## review: \n is funny: {response.isFunny}, opinion: {response.opinion}')
    return {"reviewResult": response, "retryCount": state["retryCount"] + 1}


def checkReviewResult(state: AgentState) -> Literal["genJoke", "translate"]:
    config = get_config()
    max_retries = config.get("configurable", {}).get("max_retries", 3)

    if state["reviewResult"] and state["reviewResult"].isFunny:
        print("✅ 评审通过")
        return "translate"

    if state["retryCount"] >= max_retries:
        print(f"⚠️ 达到最大重试次数 {max_retries}，强制继续")
        return "translate"

    print(f"❌ 评审未通过，重试第 {state['retryCount'] + 1}/{max_retries} 次")
    return "genJoke"


def translate(state: AgentState):
    response = structured(Joke, [{"role":"user","content":f"translate to chinese: {state['content']}"}])
    return {"content": response.joke}

## 1) Prompt Chain（顺序图）

顺序图：上游节点输出直接作为下游输入。这里 `genJoke` 生成笑话，`translate` 翻译成中文，一条线走到底。

```mermaid
graph LR
__START__ --> genJoke
genJoke --> translate
translate --> __END__
```

In [16]:
prompt_chain = StateGraph(AgentState).add_node("genJoke", genJoke)\
                              .add_node("translate", translate)\
                              .add_edge(START, "genJoke")\
                              .add_edge("genJoke", "translate")\
                              .add_edge("translate", END)
workflow = prompt_chain.compile()

# 需 API Key
result = workflow.invoke(
    {"topic": "wednesday", "content": "", "reviewResult": None, "retryCount": 0},
    {"configurable": {"thread_id": "foo"}})
print(f'#### result: \n{result}')

# gen 
 gen message: gen a Joke about wednesday


API call failed on attempt 1: Request timed out.
Max retries exceeded. Total attempts: 1, Last error: Request timed out.


InstructorRetryException: Request timed out.

## 2) Generator-Evaluator（生成-评估循环）

Generator 生产内容，Evaluator（这里是 `reviewJoke`）评审，不通过则带着意见打回 `genJoke` 重生成，直到通过或达到最大重试次数。条件边 `checkReviewResult` 决定走向。

```mermaid
graph LR
__START__ --> genJoke
genJoke --> reviewJoke
reviewJoke --reject--> genJoke
reviewJoke --approve--> translate
translate --> __END__
```

In [ ]:
graph = StateGraph(AgentState).add_node("genJoke", genJoke)\
                              .add_node("review", reviewJoke)\
                              .add_node("translate", translate)\
                              .add_edge(START, "genJoke")\
                              .add_edge("genJoke", "review")\
                              .add_conditional_edges(\
                                  "review", checkReviewResult,\
                                  {"genJoke": "genJoke", "translate": "translate"})\
                              .add_edge("translate", END)
workflow = graph.compile()

# 需 API Key；max_retries 通过 config 传入，checkReviewResult 会读到
result = workflow.invoke(
    {"topic": "wednesday", "content": "", "reviewResult": None, "retryCount": 0},
    {"configurable": {"thread_id": "foo", "max_retries": 2}})
print(f'#### result: \n{result}')

## 3) Orchestrator-Worker（并行派发）

Orchestrator 先规划出若干子任务，再用 `Send` 把每个子任务并行派发给 Worker，最后由 Synthesizer 汇总。本例 Worker 只打印、不调模型，所以**无需 API Key**，重点在理解图结构（一对多并行 + `operator.add` 累加通道）。

> ⚠️ `Send` 没有内置并发上限，Worker 会一次性全冲出去。实际调用时显式传 `config={"max_concurrency": N, ...}` 或自行做并发控制。

```mermaid
graph LR
__START__ --> orchestrator
orchestrator --> worker1
orchestrator --> worker2
worker1 --> synthesizer
worker2 --> synthesizer
synthesizer --> __END__
```

In [ ]:
from typing import Annotated, List
import operator
from pydantic import BaseModel, Field
from langgraph.graph import START, StateGraph, END
from langgraph.types import Send


class Section(BaseModel):
    name: str = Field(description="Name for this section of the report.")
    description: str = Field(description="Brief overview of the main topics and concepts to be covered in this section.")


class Sections(BaseModel):
    sections: List[Section] = Field(description="Sections of the report.")


class OWState(TypedDict):
    topic: str
    sections: list[Section]
    completed_sections: Annotated[list, operator.add]
    final_report: str


class WorkerState(TypedDict):
    section: Section
    completed_sections: Annotated[list, operator.add]


def orchestrator(state: OWState):
    """Orchestrator that generates a plan for the report"""
    task1 = Section(name="task1", description="task1")
    task2 = Section(name="task2", description="task2")
    return {"sections": [task1, task2]}


def llm_call(state: WorkerState):
    """Worker writes a section of the report"""
    print(f'working, task: {state["section"].name}, description: {state["section"].description}')
    return {"completed_sections": [f'task: {state["section"].name} finished']}


def synthesizer(state: OWState):
    """Synthesize full report from sections"""
    completed_sections = state["completed_sections"]
    completed_report_sections = "\n\n---\n\n".join(completed_sections)
    return {"final_report": completed_report_sections}


def assign_workers(state: OWState):
    """Assign a worker to each section in the plan"""
    return [Send("llm_call", {"section": s}) for s in state["sections"]]


orchestrator_worker_builder = StateGraph(OWState)
orchestrator_worker_builder.add_node("orchestrator", orchestrator)
orchestrator_worker_builder.add_node("llm_call", llm_call)
orchestrator_worker_builder.add_node("synthesizer", synthesizer)
orchestrator_worker_builder.add_edge(START, "orchestrator")
orchestrator_worker_builder.add_conditional_edges("orchestrator", assign_workers, ["llm_call"])
orchestrator_worker_builder.add_edge("llm_call", "synthesizer")
orchestrator_worker_builder.add_edge("synthesizer", END)
orchestrator_worker = orchestrator_worker_builder.compile()

result = orchestrator_worker.invoke({"topic": "Create a report on LLM scaling laws"}, config={"max_concurrency": 1})
print(f'report: {result}')

## 4) Router（分类分发）

Router 节点只做分类、不碰业务；Worker 负责生成；所有 Worker 最后汇合到同一个共用的 `translate`。下面 `genJoke` / `translate` 复用共享节点，`genKnockKnock` 是新写的另一个 Worker。

```mermaid
graph LR
__START__ --> route_joke
route_joke --person--> genJoke
route_joke --thing--> genKnockKnock
genJoke --> translate
genKnockKnock --> translate
translate --> __END__
```

In [ ]:
from typing import TypedDict, Literal
from pydantic import BaseModel, Field
from langgraph.graph import START, END, StateGraph


class Category(BaseModel):
    category: Literal["person", "thing"] = Field(description="classify the joke topic into person or thing")


# Router 状态：复用 AgentState，再多加一个分类字段
class RouterState(AgentState):
    category: str


# Router 节点：只做分类，把结果写回 state，不生成内容
def route_joke(state: RouterState) -> dict:
    response = structured(Category, [{"role": "user", "content": f"classify the topic: {state['topic']}"}])
    return {"category": response.category}


# 条件边：读 state 决定走向（与 Router 节点解耦）
def route_by_category(state: RouterState) -> Literal["genJoke", "genKnockKnock"]:
    return "genJoke" if state["category"] == "person" else "genKnockKnock"


# 新的 Worker：knock-knock 风格的笑话
def genKnockKnock(state: RouterState):
    response = structured(Joke, [{"role": "user", "content": f"tell a knock-knock joke about {state['topic']}"}])
    return {"content": response.joke}


router_builder = StateGraph(RouterState)
router_builder.add_node("route_joke", route_joke)
router_builder.add_node("genJoke", genJoke)
router_builder.add_node("genKnockKnock", genKnockKnock)
router_builder.add_node("translate", translate)
router_builder.add_edge(START, "route_joke")
router_builder.add_conditional_edges(
    "route_joke", route_by_category,
    {"genJoke": "genJoke", "genKnockKnock": "genKnockKnock"})
router_builder.add_edge("genJoke", "translate")
router_builder.add_edge("genKnockKnock", "translate")
router_builder.add_edge("translate", END)
router_graph = router_builder.compile()

# 需 API Key
result = router_graph.invoke({"topic": "wednesday", "content": "", "reviewResult": None, "retryCount": 0, "category": ""})
print(f"router result: {result}")

## 5) 嵌套子图（Nested Subgraph）

把第 2 节的「生成 → 评审 →（重试 | 翻译）」流水线，打包成一个编译好的子图 `joke_pipeline`，外层图只需把它当一个普通节点接进来。

官方文档明确：当**父图与子图共享 state key** 时，可以直接把编译后的子图传给 `add_node`，无需额外包装函数。下面父图与子图都用同一个 `AgentState`，正好满足条件。

```mermaid
graph LR
subgraph Pipeline[joke_pipeline 子图]
  G[genJoke] --> R[reviewJoke]
  R --> C{checkReviewResult}
  C --genJoke--> G
  C --translate--> T[translate]
end
__START__ --> Pipeline
Pipeline --> __END__
```

In [ ]:
from langgraph.graph import START, END, StateGraph


# 子图：复用共享的 genJoke / reviewJoke / translate / checkReviewResult
pipeline_builder = StateGraph(AgentState)
pipeline_builder.add_node("genJoke", genJoke)
pipeline_builder.add_node("reviewJoke", reviewJoke)
pipeline_builder.add_node("translate", translate)
pipeline_builder.add_edge(START, "genJoke")
pipeline_builder.add_edge("genJoke", "reviewJoke")
pipeline_builder.add_conditional_edges(
    "reviewJoke", checkReviewResult,
    {"genJoke": "genJoke", "translate": "translate"})
pipeline_builder.add_edge("translate", END)
joke_pipeline = pipeline_builder.compile()


# 父图：把子图当一个节点接进来（共享 AgentState 的 state key）
outer_builder = StateGraph(AgentState)
outer_builder.add_node("joke_pipeline", joke_pipeline)
outer_builder.add_edge(START, "joke_pipeline")
outer_builder.add_edge("joke_pipeline", END)
outer_graph = outer_builder.compile()

# 需 API Key；config 里的 max_retries 会被 checkReviewResult 读到，并自动传给子图
result = outer_graph.invoke(
    {"topic": "wednesday", "content": "", "reviewResult": None, "retryCount": 0},
    config={"configurable": {"max_retries": 2}})
print(f"nested result: {result}")

## 6) Agent 自主循环（决策 → 执行 → 回决策）

Agent 本质是一个**有上限的 LLM 循环**：LLM 思考 → 决策（调工具或直接回答）→ 执行拿到反馈 → 再思考，直到认为可以终止。

这里不引入外部工具库，而是让 LLM 返回一个结构化的「动作决策」，由一个 `call_action` 节点把决策分发到已有的 `genJoke` / `reviewJoke` / `translate`。循环次数用 `iterations` 计数兜底，达到上限强制结束，防死循环。

```mermaid
graph LR
__START__ --> agent_decide
agent_decide --> call_action
call_action --> agent_decide
agent_decide -.finish.-> __END__
```

In [ ]:
from typing import TypedDict, Optional, Literal
from pydantic import BaseModel, Field


class AgentAction(BaseModel):
    thought: str = Field(description="brief reasoning about what to do next")
    action: Literal["generate", "review", "translate", "finish"] = Field(description="next action to take")


# Agent 状态：复用 AgentState 的字段，额外加 iterations（循环计数）和 last_action（本轮决策）
class AgentLoopState(AgentState):
    iterations: int
    last_action: str


# 1) 决策节点：LLM 返回下一步动作
def agent_decide(state: AgentLoopState) -> dict:
    response = structured(AgentAction, [
            {"role": "system", "content": "You are a joke agent. Decide the next action."},
            {"role": "user", "content": f"topic={state['topic']}, current joke={state['content']}, iterations={state['iterations']}. Pick the next action."},
        ])
    return {"last_action": response.action, "iterations": state["iterations"] + 1}


# 2) 执行节点：把决策分发到已有节点（genJoke/reviewJoke/translate 全部复用）
def call_action(state: AgentLoopState) -> dict:
    action = state.get("last_action")
    if action == "generate":
        return genJoke(state)
    if action == "review":
        return reviewJoke(state)
    if action == "translate":
        return translate(state)
    return {}


# 3) 路由：finish 或达到上限就结束，否则继续循环
def route_agent(state: AgentLoopState):
    if state.get("last_action") == "finish":
        return END
    if state["iterations"] >= 5:   # 上限保护，防止死循环
        return END
    return "call_action"


agent_builder = StateGraph(AgentLoopState)
agent_builder.add_node("agent_decide", agent_decide)
agent_builder.add_node("call_action", call_action)
agent_builder.add_edge(START, "agent_decide")
agent_builder.add_conditional_edges(
    "agent_decide", route_agent,
    {"call_action": "call_action", END: END})
agent_builder.add_edge("call_action", "agent_decide")
agent_graph = agent_builder.compile()

# 需 API Key；Agent 循环会真实调用多次模型，iterations 上限（5）是兜底
result = agent_graph.invoke({
    "topic": "wednesday", "content": "", "reviewResult": None, "retryCount": 0,
    "iterations": 0, "last_action": ""})
print(f"agent result: {result}")

### 小结
- 六种模式本质是同一套积木（Node / Edge / conditional_edges / Send）的不同搭法；
- 节点天然可复用：`genJoke` / `translate` / `reviewJoke` / `checkReviewResult` 在 1/2/4/5/6 里反复出现，只定义一次；
- Orchestrator-Worker 用 `Send` 做一对多并行，配 `operator.add` 通道累加；
- 嵌套子图把一整段流程封成「节点」，外层编排层只看输入输出；
- Agent 是带上限的循环，决策权在 LLM，循环次数只是兜底保险；
- 除第 3 节外，其余都真实调用模型，跑之前确认有可用 API Key。